# NLP Project

ANTIPIN Vladislav, OUNDJIAN Milos

## Imports

In [10]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import re
from pathlib import Path
import string
import pickle


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression  # TODO we can also use this
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [12]:
from rital.data import (
    load_presidents,
    load_presidents_unseen,
    load_movies,
    load_movies_unseen,
)
from rital.preprocessing import preprocess

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data Loading

In [13]:
x_presidents, y_presidents = load_presidents()
x_presidents_unseen = load_presidents_unseen()
x_movies, y_movies = load_movies()
x_movies_unseen = load_movies_unseen()

## Basic TF-IDF Classifier

In [ ]:
# Data
# Switch this between presidents and movies
X_train = x_movies
y_train = y_movies

# TF-IDF + SVM pipeline
model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(preprocessor=preprocess)),
        ("classifier", LinearSVC()),
    ]
)

cv = StratifiedKFold(5, shuffle=True, random_state=42)

scores = cross_validate(
    model, X_train, y_train, cv=cv, scoring=["accuracy", "f1", "precision", "recall"]
)

for metric, values in scores.items():
    if "test" in metric:
        print(f"{metric}: {values.mean():.3f}")


y_pred = cross_val_predict(model, X_train, y_train, cv=cv)

print(classification_report(y_train, y_pred))

test_accuracy: 0.858
test_f1: 0.860
test_precision: 0.847
test_recall: 0.873
              precision    recall  f1-score   support

          -1       0.87      0.84      0.86      1000
           1       0.85      0.87      0.86      1000

    accuracy                           0.86      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000



## Hyperparameter Tuning

~ 1m runtime

In [ ]:
# Data
X_train = x_movies
y_train = y_movies

# CV splitter
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline
model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(preprocessor=preprocess)),
        ("classifier", LinearSVC()),
    ]
)

# Hyperparameter grid
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 5],
    "tfidf__max_df": [0.9, 1.0],
    "tfidf__sublinear_tf": [False, True],
    "classifier__C": [0.01, 0.1, 1, 10],
}

# Grid search
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",  # or "accuracy"
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_}")

best_model = grid.best_estimator_

y_pred = cross_val_predict(best_model, X_train, y_train, cv=cv)

print("Classification report:")
print(classification_report(y_train, y_pred))

Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best parameters: {'classifier__C': 1, 'tfidf__max_df': 0.9, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
Best CV score: 0.8897131878283083
Classification report:
              precision    recall  f1-score   support

          -1       0.89      0.89      0.89      1000
           1       0.89      0.89      0.89      1000

    accuracy                           0.89      2000
   macro avg       0.89      0.89      0.89      2000
weighted avg       0.89      0.89      0.89      2000



~ 1m40s runtime

In [ ]:
# Data
X_train = x_presidents
y_train = y_presidents

# CV splitter
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline
model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(preprocessor=preprocess)),
        ("classifier", LinearSVC()),
    ]
)

# Hyperparameter grid
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 5],
    "tfidf__max_df": [0.9, 1.0],
    "tfidf__sublinear_tf": [False, True],
    "classifier__C": [0.01, 0.1, 1, 10],
}

# Grid search
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",  # or "accuracy"
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_}")

best_model = grid.best_estimator_

y_pred = cross_val_predict(best_model, X_train, y_train, cv=cv)

print("Classification report:")
print(classification_report(y_train, y_pred))

Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best parameters: {'classifier__C': 1, 'tfidf__max_df': 0.9, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
Best CV score: 0.951420598118603
Classification report:
              precision    recall  f1-score   support

          -1       0.78      0.47      0.59      7523
           1       0.93      0.98      0.95     49890

    accuracy                           0.91     57413
   macro avg       0.85      0.73      0.77     57413
weighted avg       0.91      0.91      0.90     57413



# Word2Vec weighted by TF-IDF

In [ ]:
import gensim.downloader as api

# Load the model
glove_embeddings = api.load("glove-twitter-200")

# Other Classifiers

In [7]:
X_train = x_movies
y_train = np.array(y_movies)

In [50]:
SCORE = "f1"
# 1 is expected to be minority class, need to balance classes where possible
ratio = np.sum(y_train == 0) / np.sum(y_train == 1)

# Preprocess ========================
preprocess_param_grid = {
    "tfidf__ngram_range": [(1,2)],#[(1, 1), (1, 2)],
    "tfidf__min_df": [2],#[1, 2, 5],
    "tfidf__max_df": [0.9],#[0.9, 1.0],
    "tfidf__sublinear_tf": [True]#[False, True],
}

# Linear SVM =================================================================================

svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=preprocess)),
    ("svm", SVC(class_weight='balanced'))
    ])

svm_param_grid = [
    {
        **preprocess_param_grid,
        "svm__kernel": ["linear"],
        "svm__C": [0.01, 0.1, 1, 10]
    },
    {
        **preprocess_param_grid,
        "svm__kernel": ["poly"],
        "svm__C": [0.1, 1, 10],
        "svm__degree": [2, 3, 4],
        "svm__gamma": ["scale", "auto"]
    },
    {
        **preprocess_param_grid,
        "svm__kernel": ["rbf"],
        "svm__C": [0.1, 1, 10],
        "svm__gamma": ["scale", "auto"]
    }
]

svm_cv = GridSearchCV(svm_model, svm_param_grid, cv=5, scoring=SCORE, n_jobs=-1, verbose=3)

# Random Forest =============================================================================
rf_model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(preprocessor=preprocess)),
    ("rf", RandomForestClassifier(class_weight='balanced',n_estimators=300))
])
rf_param_grid = {
    **preprocess_param_grid,
    "rf__n_estimators": [100, 300, 500],
    "rf__max_depth": [None, 10, 20],
    "rf__min_samples_split": [2, 5],
    "rf__min_samples_leaf": [1, 2]
}

rf_cv = GridSearchCV(rf_model, rf_param_grid, cv=5, scoring=SCORE, n_jobs=-1, verbose=3)

# XGBoost ==========================================================================


xgb_model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(preprocessor=preprocess)),
    ("xgb", XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        # use_label_encoder=False,
        # scale_pos_weight= ratio # NOTE: may need for presidents
    ))
])

xgb_param_grid = {
    **preprocess_param_grid,
    "xgb__n_estimators": [100, 300],
    "xgb__max_depth": [3, 5],
    "xgb__learning_rate": [0.01, 0.05, 0.1],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0], 
    "xgb__gamma": [0, 0.1]  
}

xgb_cv = GridSearchCV(xgb_model, xgb_param_grid, cv=5, scoring=SCORE, n_jobs=-1, verbose=3)

# All models =======================================================================================
model_cvs = (
    rf_cv,
    xgb_cv,
    # svm_cv,
    
)

full_names = {
    "xgb" : "XGBoost",
    # "svm": "Support Vector Machine",
    "rf": "Random Forest",
    
}

In [51]:
RUN_TUNING = True
MODEL_PATH = "../models/"
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
if RUN_TUNING:
    summary_df = pd.DataFrame()
    models = {}
    for model_cv in model_cvs:
        model_cv.fit(X_train, y_train)
        # NOTE: best estimator is trained on the whole train dataset
        model = model_cv.best_estimator_ 
        model_name = list(model.named_steps.keys())[1]
        models[model_name] = model
        cv_df = pd.DataFrame(model_cv.cv_results_)[['params', 'mean_test_score', 'std_test_score']]
        cv_df = cv_df[cv_df["mean_test_score"] == cv_df["mean_test_score"].max() ]

        model.fit(X_train, y_train)
        
        print(model_name, ":::::::::::::::::::::::::::::::::::")
        
        print(f"Best parameters: {model_cv.best_params_}")
        print(f"Best CV score: {model_cv.best_score_}")

        y_pred = cross_val_predict(model, X_train, y_train, cv=cv)

        print("Classification report:")
        print(classification_report(y_train, y_pred))

        summary_row = pd.DataFrame([{"model" : list(model.named_steps.keys())[1]}])
        summary_row.reset_index(drop=True, inplace=True)
        cv_df.reset_index(drop=True, inplace=True)
        summary_row = pd.concat([summary_row, cv_df], axis=1)
        summary_df = pd.concat([summary_df, summary_row], axis=0)
        
    summary_df.reset_index(drop=True, inplace=True)
    with open(MODEL_PATH + "summary_df.pkl","wb") as f:
            pickle.dump(summary_df,f)
    with open(MODEL_PATH + "models.pkl","wb") as f:
        pickle.dump(models,f)
else:
    with open(MODEL_PATH + "summary_df.pkl","rb") as f:
        summary_df =pickle.load(f)
    with open(MODEL_PATH + "models.pkl","rb") as f:
        models =pickle.load(f)
    

Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV 2/5] END rf__max_depth=None, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=100, tfidf__max_df=0.9, tfidf__min_df=2, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True;, score=0.725 total time=   4.8s
[CV 4/5] END rf__max_depth=None, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=100, tfidf__max_df=0.9, tfidf__min_df=2, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True;, score=0.754 total time=   4.9s
[CV 3/5] END rf__max_depth=None, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=100, tfidf__max_df=0.9, tfidf__min_df=2, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True;, score=0.773 total time=   4.9s
[CV 1/5] END rf__max_depth=None, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=100, tfidf__max_df=0.9, tfidf__min_df=2, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True;, score=0.768 total time=   5.2s
[CV 5/5] END rf__max_depth=None, rf__m

KeyboardInterrupt: 